<a href="https://colab.research.google.com/github/its-asif/RAG-ask-from-youtube-video/blob/main/Rag_youtube_transcription_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
from google.colab import userdata

def get_secret(name: str) -> str:
    val = userdata.get(name)
    if val:
      return val

groq_key = get_secret("GROQ_API_KEY")
supadata_key = get_secret("SUPADATA_API_KEY")

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-openai langchain_groq langchain_huggingface \
                faiss-cpu tiktoken python-dotenv langchain-text-splitters langchain_huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

/tmp/ipykernel_7601/1898699603.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
!pip install supadata

# Step 1 - Indexing


## Document Ingestion

In [5]:
# https://www.youtube.com/watch?v=fxx_E0ojKrc"

In [6]:
import os
import re
import json
from google.colab import drive, userdata
from supadata import Supadata

In [7]:
# 1. Mount Google Drive for persistent storage across Colab sessions
drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/youtube_transcripts"
os.makedirs(CACHE_DIR, exist_ok=True)

Mounted at /content/drive


In [8]:
# 2. Initialize Supadata
supadata = Supadata(api_key = supadata_key)

def extract_video_id(url: str) -> str:
    """Extracts the 11-character YouTube video ID from standard, short, or embed URLs."""
    match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11})(?:[&?\/]|$)", url)
    if not match:
        raise ValueError(f"Could not extract a valid YouTube video ID from: {url}")
    print(match.group(1))
    return match.group(1)

def get_transcript_cached(url: str, lang: str = "en") -> dict:
    video_id = extract_video_id(url)
    file_path = os.path.join(CACHE_DIR, f"{video_id}.json")

    # 3. Return cached file if it already exists
    if os.path.exists(file_path):
        print(f"[CACHE HIT] Loaded {video_id}.json from Google Drive")
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)

    # 4. Otherwise fetch from Supadata and save to Drive
    print(f"[API CALL] Fetching {video_id} from Supadata...")
    result = supadata.transcript(url=url, lang=lang)
    print(result)

    # Serialize the SDK response object to a clean JSON-compatible dict
    data = {
        "video_id": video_id,
        "url": url,
        "lang": result.lang,
        "available_langs": result.available_langs,
        "content": [
            {
                "text": chunk.text,
                "offset": chunk.offset,
                "duration": chunk.duration,
                "lang": chunk.lang,
            }
            for chunk in result.content
        ],
    }

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[SAVED] Cached to {file_path}")
    return data

# --- Example Usage ---
transcript_data = get_transcript_cached("https://www.youtube.com/watch?v=fxx_E0ojKrc")
print(f"Total chunks: {len(transcript_data['content'])}")

fxx_E0ojKrc
[CACHE HIT] Loaded fxx_E0ojKrc.json from Google Drive
Total chunks: 2408


In [9]:
transcript_data['content'][:10]

[{'text': 'This course is an introduction to time',
  'offset': 0,
  'duration': 4720,
  'lang': 'en'},
 {'text': "series forecasting with Python. You'll",
  'offset': 1920,
  'duration': 5160,
  'lang': 'en'},
 {'text': 'learn what time series data is and how',
  'offset': 4720,
  'duration': 4720,
  'lang': 'en'},
 {'text': 'to break it down into its key components',
  'offset': 7080,
  'duration': 5720,
  'lang': 'en'},
 {'text': 'like trend, seasonality, and residuals.',
  'offset': 9440,
  'duration': 5240,
  'lang': 'en'},
 {'text': "You'll start by building simple baseline",
  'offset': 12800,
  'duration': 4160,
  'lang': 'en'},
 {'text': 'models before learning about powerful',
  'offset': 14680,
  'duration': 5760,
  'lang': 'en'},
 {'text': 'forecasting techniques such as ARIMA and',
  'offset': 16960,
  'duration': 6120,
  'lang': 'en'},
 {'text': "seasonal ARIMA. You'll discover how to",
  'offset': 20440,
  'duration': 5040,
  'lang': 'en'},
 {'text': 'forecast future val

In [10]:
transcript_full_text = " ".join(item["text"].strip() for item in transcript_data["content"])
transcript_full_text

'This course is an introduction to time series forecasting with Python. You\'ll learn what time series data is and how to break it down into its key components like trend, seasonality, and residuals. You\'ll start by building simple baseline models before learning about powerful forecasting techniques such as ARIMA and seasonal ARIMA. You\'ll discover how to forecast future values, evaluate your models using cross-validation, and incorporate exogenous features to improve predictions. The course also covers how to generate prediction intervals and select the most appropriate evaluation metrics for your projects. You\'ll gain a clear understanding of the main forecasting techniques and how to apply them in practice. And you\'re in good hands. Marco teaches this course and he wrote an entire book on the topic. >> Hello and welcome to this video. If you are new here, let me just quickly introduce myself. My name is Marco and I am passionate about teaching and about time series forecasting.

Indexing (Text Splitting)

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_full_text])

In [12]:
len(chunks)

104

In [13]:
chunks[100]

Document(metadata={}, page_content="that SARIMA is better. Scaled CRPS also um useful when you are doing uh forecast with prediction intervals. You get that SARIMA is better. In this case, the MAPE and sMAPE not very useful because we have data that is close to zero, so we shouldn't be looking at them. Uh the MAE, it's okay. However, do notice that your series have different uh scales, okay? So, there might be a bit of an uh of a bias in there. MSE, RMSE, in my opinion, the MSE is hard to interpret. The RMSE uh is fine. Uh however, you still have this uh scale problem, and you are uh giving more importance to large errors, whether it's something that you want or not, this is up to you to decide. So, that's it. This is how you can evaluate your models using different metrics. Congratulations on making it this far. You have done a lot. You have learned a lot as well. I am super proud of you. I hope that you are proud of yourself. But, now what's next? Okay, where do we go from here? So, 

Indexing (Embedding Generation and Storing in Vector Store)

In [14]:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
vector_store.index_to_docstore_id

{0: '5dd66e74-f6a9-48a9-b0ab-cc64a64690da',
 1: '2c43b0de-31d4-4e61-9574-4420ed0efe0c',
 2: '104dbaf3-3a92-4192-bec3-99b9ea3daa08',
 3: 'b18c90d9-762c-447e-a575-eeda783c06a6',
 4: '5b73c157-6b1b-4083-b9c2-1523beaed4c1',
 5: '36f1cb77-1e0b-42af-bdc0-6d521a9b9c5c',
 6: '981a2f51-a95d-4ce1-911f-62b69a70d3ab',
 7: 'fcced40d-9f1e-46f8-a27b-9a97c7d8528e',
 8: '85823b69-5c44-4cbd-93de-cc56c547fbbd',
 9: 'b01bf34d-93ec-47d7-a7d2-f7ad930dcfa3',
 10: '9b262bd7-18ad-4d95-b63a-4487063e5035',
 11: '07bab0b5-5ed4-4b83-b40a-a217e4a431ed',
 12: '42a33d34-698b-4e1d-b203-12238f609a13',
 13: 'c94d1cd9-5ad4-4612-9aed-4ff9f754809c',
 14: 'e1b02bea-0a43-4f81-940e-140d1be44861',
 15: '0f162087-6f36-4668-a312-c7afc4838a19',
 16: 'e5f7bcbf-ff3f-4c08-8144-03b492e12f02',
 17: 'b1f47548-e789-4083-af6a-70c42cdac225',
 18: 'fabaea06-2ad3-43ca-b902-9725eeb888a6',
 19: 'b3809eae-325a-4048-90cb-3fbf86abfaf9',
 20: '98bd6cf1-d370-4d3c-af7f-249776a3e416',
 21: 'e9aaadde-9a2b-4b9d-a75f-80efb95ccedf',
 22: '5edf4d80-6172-

## Step 2 - Retrieval

In [16]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f6240392e40>, search_kwargs={'k': 4})

In [17]:
retriever.invoke('How to calculate time series')

[Document(id='5b73c157-6b1b-4083-b9c2-1523beaed4c1', metadata={}, page_content="right? Generating prediction intervals, and finally, we are going to see how we can evaluate those forecasting models. And so, as you can see from this video, we're going to get a complete picture of forecasting. Only here we are focusing mostly on using statistical models, and of course, everything will be coded in Python. So, let's start by defining time series, all right? And a time series is super simple. It's a set of data points ordered in time. And ideally, the data is equally spaced in time, so you have, you know, data every minute or every hour. And so, there are many examples of time series, right? That we see day-to-day. For example, the closing price of stock, the electricity consumption of a of a household, or even the temperature outside, okay? So, as long as your data is indexed in time, it can be viewed as a time series. And here, for example, I show you an example of time series, which in t

## Step 3 - Augmentation

In [18]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key = groq_key
)

In [19]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [20]:
question          = "is the process of calculating time series discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)
retrieved_docs

[Document(id='5b73c157-6b1b-4083-b9c2-1523beaed4c1', metadata={}, page_content="right? Generating prediction intervals, and finally, we are going to see how we can evaluate those forecasting models. And so, as you can see from this video, we're going to get a complete picture of forecasting. Only here we are focusing mostly on using statistical models, and of course, everything will be coded in Python. So, let's start by defining time series, all right? And a time series is super simple. It's a set of data points ordered in time. And ideally, the data is equally spaced in time, so you have, you know, data every minute or every hour. And so, there are many examples of time series, right? That we see day-to-day. For example, the closing price of stock, the electricity consumption of a of a household, or even the temperature outside, okay? So, as long as your data is indexed in time, it can be viewed as a time series. And here, for example, I show you an example of time series, which in t

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"right? Generating prediction intervals, and finally, we are going to see how we can evaluate those forecasting models. And so, as you can see from this video, we're going to get a complete picture of forecasting. Only here we are focusing mostly on using statistical models, and of course, everything will be coded in Python. So, let's start by defining time series, all right? And a time series is super simple. It's a set of data points ordered in time. And ideally, the data is equally spaced in time, so you have, you know, data every minute or every hour. And so, there are many examples of time series, right? That we see day-to-day. For example, the closing price of stock, the electricity consumption of a of a household, or even the temperature outside, okay? So, as long as your data is indexed in time, it can be viewed as a time series. And here, for example, I show you an example of time series, which in this case tracks the monthly milk production in Australia. So, every month we're

In [22]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      right? Generating prediction intervals, and finally, we are going to see how we can evaluate those forecasting models. And so, as you can see from this video, we're going to get a complete picture of forecasting. Only here we are focusing mostly on using statistical models, and of course, everything will be coded in Python. So, let's start by defining time series, all right? And a time series is super simple. It's a set of data points ordered in time. And ideally, the data is equally spaced in time, so you have, you know, data every minute or every hour. And so, there are many examples of time series, right? That we see day-to-day. For example, the closing price of stock, the electricity consumption of a of a household, or even the temperature outside, okay? So, as long as your data is indexed in t

## Step 4 - Generation

In [23]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes. The video walks through the whole **process of working with (and forecasting) a time‑series**, covering the following steps:

1. **Definition of a time series** – a sequence of data points ordered in time, ideally equally spaced (e.g., minute‑by‑minute, hourly, monthly).  
2. **Components of a time series** –  
   * **Trend** (long‑term increase or decrease)  
   * **Seasonality** (repeating pattern at fixed intervals)  
   * The original series is shown together with its decomposed components.  
3. **Baseline models** – simple reference models that are built first to have a performance benchmark.  
4. **ARIMA modeling** – introduction to the classic statistical forecasting model (AutoRegressive Integrated Moving Average).  
5. **More advanced techniques** –  
   * **Cross‑validation** for time‑series data  
   * **Using exogenous (external) features** in the model  
   * **Generating prediction intervals** to quantify forecast uncertainty  
   * **Evaluating forecasting models** 